# Object tection using Deep Learning

Dataset link: https://www.kaggle.com/datasets/sshikamaru/car-object-detection

Required libraries

In [ ]:
!pip install pytorch

  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pytorch
  Running setup.py clean for pytorch
Failed to build pytorch
ERROR: Could not build wheels for pytorch, which is required to install pyproject.toml-based projects


In [ ]:
import numpy as np
import pandas as pd
import os
import random
from PIL import Image, ImageDraw
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torchvision
from torchvision import transforms as T
from torchvision.models.detection. faster_rcnn import FastRCNNPredictor

NameError: ignored

1. read the training data

In [ ]:
train= pd.read_csv("/content/data/train_solution_bounding_boxes (1).csv")

In [ ]:
train.head()

,image,xmin,ymin,xmax,ymax
0,vid_4_1000.jpg,281.259045,187.035071,327.727931,223.225547
1,vid_4_10000.jpg,15.163531,187.035071,120.329957,236.430180
2,vid_4_10040.jpg,239.192475,176.764801,361.968162,236.430180
3,vid_4_10020.jpg,496.483358,172.363256,630.020260,231.539575
4,vid_4_10060.jpg,16.630970,186.546010,132.558611,238.386422


In [ ]:
unique_imgs=train.image.unique()

In [ ]:
unique_imgs

array(['vid_4_1000.jpg', 'vid_4_10000.jpg', 'vid_4_10040.jpg',
       'vid_4_10020.jpg', 'vid_4_10060.jpg', 'vid_4_10100.jpg',
       'vid_4_10120.jpg', 'vid_4_10140.jpg', 'vid_4_1020.jpg',
       'vid_4_1040.jpg', 'vid_4_10480.jpg', 'vid_4_10500.jpg',
       'vid_4_10520.jpg', 'vid_4_1060.jpg', 'vid_4_10960.jpg',
       'vid_4_10980.jpg', 'vid_4_11000.jpg', 'vid_4_11020.jpg',
       'vid_4_11240.jpg', 'vid_4_11260.jpg', 'vid_4_11280.jpg',
       'vid_4_11380.jpg', 'vid_4_11400.jpg', 'vid_4_11420.jpg',
       'vid_4_11440.jpg', 'vid_4_11900.jpg', 'vid_4_11880.jpg',
       'vid_4_11920.jpg', 'vid_4_11940.jpg', 'vid_4_11960.jpg',
       'vid_4_11980.jpg', 'vid_4_12000.jpg', 'vid_4_12040.jpg',
       'vid_4_12100.jpg', 'vid_4_12060.jpg', 'vid_4_12080.jpg',
       'vid_4_12120.jpg', 'vid_4_12140.jpg', 'vid_4_12160.jpg',
       'vid_4_12180.jpg', 'vid_4_12200.jpg', 'vid_4_12220.jpg',
       'vid_4_12240.jpg', 'vid_4_12260.jpg', 'vid_4_12280.jpg',
       'vid_4_12300.jpg', 'vid_4_12320.jpg',

In [ ]:
class CustDat(torch.utils.data.Dataset):
  def __init__(self, df, unique_imgs, indices):
    self.df = df
    self.unique_imgs = unique_imgs
    self.indices= indices

  def __len__(self):
    return len(self.indices)

  def __getitem__(self, idx):
    image_name= self.unique_imgs[self.indices[idx]]
    boxes= self.df[self.df.image == image_name].values[:, 1:].astype("float")
    img = Image.open("/content/data/training_images/" + image_name ).convert('RGB')
    labels = torch.ones((boxes.shape[0]), dtype = torch.int64)
    target ={}
    target["boxes"] = torch.tensor (boxes)
    target["label"] = labels
    return T.ToTensor() (img), target

NameError: ignored

In [ ]:
train_inds,val_inds=train_test_split(range(unique_imgs.shape[0]),test_size=0.1)

In [ ]:
unique_imgs[val_inds]

In [ ]:
def custom_collate(data):
  return data

In [ ]:
train_dl=torch.utils.data.DataLoader(CustDat(train,unique_imgs,train_inds),
                                     batch_size=8,
                                     shuffle=True,
                                     collate_fn=custom_collate,
                                     pin_memory=True if torch.cuda.is_available() else False)

val_dl=torch.utils.data.DataLoader(CustDat(train,unique_imgs,val_inds),
                                     batch_size=8,
                                     shuffle=True,
                                     collate_fn=custom_collate,
                                     pin_memory=True if torch.cuda.is_available() else False)

In [ ]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained = True)
num_classes = 2
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

In [ ]:
device= torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

In [ ]:
device

In [ ]:
optimizer= torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9, weight_decay=0.0005)
num_epochs=10

In [ ]:
model.to(device)
for epochs in range (num_epochs):
    epoch_loss = 0
    best_metric = float('inf')
    best_epoch = -1
    for data in train_dl:
        imgs = []
        targets = []
        for d in data:
            imgs.append(d[0].to(device))
            targ= {}
            targ["boxes"] = d[1]["boxes"].to(device)
            targ["labels"] = d[1]["label"].to(device)
            targets.append(targ)
        loss_dict = model(imgs, targets)
        loss = sum(v for v in loss_dict.values())
        epoch_loss += loss.cpu().detach().numpy()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch_loss < best_metric:
        # Update the best metric and best epoch
            best_metric = epoch_loss
            best_epoch = epochs
            # Save the model with the current best performance
            torch.save(model.state_dict(), "best_model.pth")
    print(epoch_loss)


In [ ]:
model.eval()
data=iter(val_dl).__next__()

visualise the training data

Analysis of the data

In [ ]:
#no of image that ahave car  and nover that doesnt

split into train and validation

create a CNN model

fit the model

interprete with test dataset

1. load test data
2. transform test data
3. show test data
4. model. predict (test data)

model accuracy, evaluatioan matrix